In [0]:
# %sql
# select * from table s
# join table t 
# join table2 t2 
# on s.c1 = t.c1  and s.c2 = t.c2 
#  and s.c3 = t2.c3 

In [0]:
# %sql
# select s.id, t.id from table s
# join table t 
# join table2 t2 
# on s.c1 = t.c1  and s.c2 = t.c2 
#  and s.c3 = t2.c3 

In [0]:
df = spark.sql("""
select * from formula1_dev.silver. results s
join table t 
join table2 t2 
on s.c1 = t.c1  and s.c2 = t.c2 
 and s.c3 = t2.c3 
""")

In [0]:
catalog = f"formula1_dev"
results_df = spark.table(f"{catalog}.silver.results")
races_df = spark.table(f"{catalog}.silver.races")
constructors_df=spark.table(f"{catalog}.silver.constructors")
                      

In [0]:
from pyspark.sql.functions import col
final_df= results_df.alias("res").join(races_df.alias("rac"), col("res.race_id") == col("rac.race_id"), "inner" )\
    .join(constructors_df.alias("con"), col("res.constructor_id") == col("con.constructor_id"), "left" )
# final_df= results_df.alias("res").join(races_df.alias("rac"), col("res.race_id") == col("rac.race_id"), "left" ) #right, full ,left semi , left anti
final_df.display()

In [0]:
gold_df= final_df.select(col("res.result_id"),col("res.race_id"), col("res.constructor_id"), col("rac.race_year"), col("position"), col("position_order"), col("position_text"), col("grid"), col("points"))
gold_df.display()

In [0]:
from pyspark.sql.functions import when,col
gold_df_final= gold_df.withColumn("finish_group", when(col("position_order")== 1, "Winner")
                                  .when(col("position_order")== 2, "Runner Up")
                                  .when(col("position_order")<= 3, "top_3")
                                  .when(col("position").isNotNull(), "outside_top_3")
                                  .otherwise("No_final_position")
)
gold_df_final.display()

In [0]:
from pyspark.sql.functions import current_timestamp
gold_df_final_1=gold_df_final.withColumn("gold_ingestion_timestamp", current_timestamp())

In [0]:
gold_df_final_1.write.mode("overwrite").format("delta").saveAsTable("formula1_dev.gold.results_driver_performance")